# Imports and definitions

In [ ]:
import os
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
import seaborn as sns

from glob import glob
from nilearn.image import resample_to_img, binarize_img
from nilearn.masking import apply_mask
from nilearn import plotting

from scipy.stats import ttest_1samp, ttest_ind
from statsmodels.stats.anova import AnovaRM
from statsmodels.stats.multitest import multipletests
import pingouin as pg

from stats_fmt import export_ttests


In [ ]:
pd.set_option("display.float_format", '{:.4f}'.format)


In [ ]:
task_label = 'badaga'
fwhm = 6.00
space_label = 'MNI152NLin2009cAsym'

# define data directories
bidsroot = os.path.join('/bgfs/bchandrasekaran/krs228/data/',
                        'SSP/',
                        'data_bids')
nilearn_dir = os.path.join(bidsroot, 'derivatives', 'nilearn')
masks_dir = os.path.join(nilearn_dir, 'masks')
# define first-level directory where group-level inputs will be pulled from
l1_dir = os.path.join(nilearn_dir, 'run-all_contrast-snr')

# create output directory
group_out_dir = os.path.join(nilearn_dir, 'group_fwhm-%.02f' % fwhm)
os.makedirs(group_out_dir, exist_ok=True)


In [ ]:
bidsroot


In [ ]:
contrast_list = ['q', '8', '0', 'n2', 'n6']

# Cortical ROIs for the univariate ROI analysis: 14 "auditory" ROIs (Heschl's gyrus, planum
# temporale/polare, superior temporal gyrus posterior/anterior, pars opercularis/triangularis,
# bilaterally) plus 6 "extended language network" ROIs (supramarginal gyrus anterior/posterior +
# angular gyrus, bilaterally) -- same 20-ROI list used in the GLMsingle RSA pipeline
# (GLMsingle_mask-betas.py / GLMsingle_rsa-roi.py / GLMsingle_rsa-group.ipynb), kept in sync
# across all files per this repo's established convention. Ordered all-L-then-all-R (mirrored)
# rather than per-network L/R blocks, for easier hemisphere comparisons in downstream plots.
CORTICAL_ROI_LIST = [
    'L-HG', 'L-PT', 'L-PP', 'L-STGp', 'L-STGa', 'L-ParsOp', 'L-ParsTri', 'L-SMGa', 'L-SMGp', 'L-Ang',
    'R-HG', 'R-PT', 'R-PP', 'R-STGp', 'R-STGa', 'R-ParsOp', 'R-ParsTri', 'R-SMGa', 'R-SMGp', 'R-Ang',
]

# base region order (no hemisphere prefix), for the hemisphere-hue group-specific plots below
BASE_REGION_ORDER = [roi.split('-', 1)[1] for roi in CORTICAL_ROI_LIST if roi.startswith('L-')]


Based on [nilearn documentation](https://nilearn.github.io/stable/auto_examples/05_glm_second_level/plot_thresholding.html#statistical-testing-of-a-second-level-analysis)

### Build the group-level design matrix

#### Read the `participants.tsv` file from the BIDS root directory

In [ ]:
participants_fpath = os.path.join(bidsroot, 'participants.tsv')
participants_df = pd.read_csv(participants_fpath, sep='\t')

# subjects to ignore (not fully processed, etc.)
ignore_subs = ['sub-SSP001', 'sub-SSP002',
               'sub-SSP005', 'sub-SSP012',
               'sub-SSP014',
               'sub-SSP069', 'sub-SSP072',
               'sub-SSP076', # missing MRI files
               'sub-SSP102', # participant left after a few minutes
               'sub-SSP098', 'sub-SSP105', 'sub-SSP106', 'sub-SSP107', # currently processing
               ]
participants_df.drop(participants_df[participants_df.participant_id.isin(ignore_subs)].index, inplace=True)

# re-sort by participant ID
participants_df.sort_values(by=['participant_id'], ignore_index=True, inplace=True)


In [ ]:
print(participants_df)


### Group membership

In [ ]:
# group labels in participants.tsv have been observed with inconsistent casing (e.g. 'CWS' vs
# 'cws') -- compare against a case- and whitespace-normalized copy rather than the raw strings,
# same fix already applied in univariate_group-level.ipynb.
group_norm = participants_df.group.str.strip().str.lower()
sub_list_cwns = list(participants_df.participant_id[group_norm == 'control'])
sub_list_cws = list(participants_df.participant_id[group_norm == 'cws'])

participants_cwns_df = participants_df[group_norm == 'control'].reset_index(drop=True)
participants_cws_df = participants_df[group_norm == 'cws'].reset_index(drop=True)

group_lookup = {**{s: 'CWNS' for s in sub_list_cwns}, **{s: 'CWS' for s in sub_list_cws}}

print(f'CWNS: {len(sub_list_cwns)} subjects')
print(f'CWS: {len(sub_list_cws)} subjects')


### Define key processing functions

In [ ]:
def build_statmap_dict(sub_list, contrast_list, l1_dir):
    """Subject-ID-keyed {contrast: {sub_id: path}} -- a missing file is skipped (logged) rather
    than a bare glob(...)[0] IndexError, mirroring prepare_group_inputs' graceful-drop convention
    in univariate_group-level.ipynb.
    """
    statmap_dict = {}
    for contrast_label in contrast_list:
        contrast_dict = {}
        for sub_id in sub_list:
            pattern = os.path.join(l1_dir, sub_id, f'*contrast-{contrast_label}_stat-effect_statmap.nii.gz')
            matches = sorted(glob(pattern))
            if not matches:
                print(f'No contrast-{contrast_label} statmap found for {sub_id} -- skipping.')
                continue
            contrast_dict[sub_id] = matches[0]
        statmap_dict[contrast_label] = contrast_dict
    return statmap_dict


In [ ]:
def mask_stat_maps(roi_list, statmap_dict, masks_dir, space_label):
    """Returns {roi: {sub_id: mean_beta}} -- subject-ID-keyed (not positional lists) so a missing
    mask for one subject/ROI can be skipped (logged) without misaligning any other ROI's or
    subject's values, instead of the previous bare glob(...)[0] IndexError.
    """
    roi_mean_dict = {}
    for roi in roi_list:
        print(roi)
        sub_mean_dict = {}
        for sub_id, stat_fpath in statmap_dict.items():
            mask_pattern = os.path.join(masks_dir, sub_id, f'space-{space_label}', 'masks-dseg', f'*{roi}*.nii.gz')
            matches = glob(mask_pattern)
            if not matches:
                print(f'No mask found for {sub_id}, ROI {roi} -- skipping.')
                continue
            mask_fpath = matches[0]

            mask_img = resample_to_img(mask_fpath, stat_fpath,
                                       interpolation='nearest',
                                       force_resample=True,
                                       copy_header=True)
            mask_img = binarize_img(mask_img, two_sided=False, copy_header=True)
            masked_data = apply_mask(stat_fpath, mask_img)
            sub_mean_dict[sub_id] = masked_data.mean()

        roi_mean_dict[roi] = sub_mean_dict

    return roi_mean_dict


In [ ]:
def make_stats_df(roi_mean_dict, group_lookup):
    """Build a long-format ROI dataframe from {roi: {sub_id: mean_beta}} (subject-ID-keyed, not
    positional lists) -- avoids assuming roi_mean_dict's row order matches any other dataframe's
    row order, since mask_stat_maps may skip a subject/ROI combination with a missing mask.
    """
    records = []
    for region_hemi, sub_mean_dict in roi_mean_dict.items():
        for sub_id, beta in sub_mean_dict.items():
            records.append({
                'participant_id': sub_id,
                'group': group_lookup[sub_id],
                'region_hemi': region_hemi,
                'beta': beta,
            })
    roi_df_long = pd.DataFrame(records)
    roi_df_long['hemisphere'] = roi_df_long.region_hemi.str.split('-', n=1).str[0]
    roi_df_long['region'] = roi_df_long.region_hemi.str.split('-', n=1).str[1]

    return roi_df_long


### Box + strip plotting helpers

Mirrors `multivariate_fmri/GLMsingle_rsa-group.ipynb`'s `plot_roi_box_strip` pattern: stripplot +
boxplot overlay, both with fill stripped to transparent and outlined in each hue group's assigned
color instead. ROIs with a significant within-group effect (FDR-corrected p < 0.05, from the
statistics section below) get an asterisk above them, with headroom added so it doesn't collide
with the title.

Two variants:
1. **Combined** (`x=region_hemi` for all 20 ROIs, `hue=group`) -- one figure per contrast. Asterisk
   marks a ROI if *either* group is significant there (not group-specific).
2. **Group-specific** (`x=region` -- base ROI name, hemisphere prefix stripped -- `hue=hemisphere`)
   -- one figure per (contrast, group), fewer x-categories (10 base regions instead of 20 ROIs).
   Asterisks are placed over the specific L or R box (using seaborn's dodge offset).

In [ ]:
def _outline_only(ax):
    """Strip fill from stripplot dots and boxplot boxes, re-applying each hue group's original
    fill color as the outline/edge color first -- so the now-transparent shapes are still
    outlined in the color they're assigned, rather than whatever default edge color seaborn
    would otherwise leave them with.
    """
    for collection in ax.collections:
        facecolor = collection.get_facecolor()
        collection.set_edgecolor(facecolor)
        collection.set_facecolor('none')
    for patch in ax.patches:
        facecolor = patch.get_facecolor()
        patch.set_edgecolor(facecolor)
        patch.set_facecolor('none')


def _add_asterisk_headroom(ax, y_top):
    """Compute an asterisk y-position with headroom above the highest data point, and expand
    the axes' y-limit to guarantee that headroom doesn't get clipped or collide with the title.
    Uses a fixed fraction of the actual visible y-range, so it works correctly regardless of
    whether y_top is positive, negative, or near zero (unlike a `y_top * 1.05`-style formula).
    """
    y_min, y_max = ax.get_ylim()
    y_range = y_max - y_min
    asterisk_y = y_top + 0.08 * y_range
    ax.set_ylim(y_min, y_max + 0.15 * y_range)
    return asterisk_y


def plot_roi_box_strip(roi_df_long, sig_df, contrast_label, roi_order):
    contrast_df = roi_df_long[roi_df_long.SNR == contrast_label]

    fig, ax = plt.subplots(1, 1, figsize=(0.5 * len(roi_order) + 2, 4), dpi=300)

    sns.stripplot(data=contrast_df, x='region_hemi', y='beta', hue='group', order=roi_order,
                 dodge=True, linewidth=0.5, size=3, legend=None, ax=ax, zorder=2)
    sns.boxplot(data=contrast_df, x='region_hemi', y='beta', hue='group', order=roi_order,
               dodge=True, linewidth=1, fliersize=0, ax=ax, zorder=1)
    _outline_only(ax)

    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    sns.move_legend(ax, 'upper left', bbox_to_anchor=(1, 1), title='Group')
    ax.axhline(y=0, color='0.5', linestyle='--', linewidth=0.5)
    ax.set_ylabel('mean beta')
    ax.set_title(f'contrast-{contrast_label}')

    # mark ROIs with ANY significant within-group effect (either group) with an asterisk
    y_top = contrast_df['beta'].max()
    asterisk_y = _add_asterisk_headroom(ax, y_top)
    sig_rois = sig_df[(sig_df.SNR == contrast_label) & (sig_df.p_fdr < 0.05)]['region_hemi'].unique()
    for roi in sig_rois:
        if roi in roi_order:
            ax.text(roi_order.index(roi), asterisk_y, '*', ha='center', va='bottom',
                    fontsize=14, fontweight='bold')

    fig.tight_layout()
    sns.despine(ax=ax)
    return fig


# Seaborn's default dodge for 2 hue categories with the default box width (0.8) centers each
# hue's sub-box at +/-0.2 from the category tick position.
HEMISPHERE_ORDER = ['L', 'R']
HEMISPHERE_OFFSET = {'L': -0.2, 'R': 0.2}


def plot_roi_box_strip_by_hemisphere(roi_df_long, sig_df, contrast_label, group_name, region_order):
    """Group-specific variant: x=region (hemisphere prefix stripped), hue=hemisphere. Asterisks
    are placed over the specific L or R box (not the region's center), so significance is shown
    per-hemisphere, not pooled across both.
    """
    group_df = roi_df_long[(roi_df_long.SNR == contrast_label) & (roi_df_long.group == group_name)]

    fig, ax = plt.subplots(1, 1, figsize=(0.5 * len(region_order) + 2, 4), dpi=300)

    sns.stripplot(data=group_df, x='region', y='beta', hue='hemisphere', order=region_order,
                 hue_order=HEMISPHERE_ORDER,
                 dodge=True, linewidth=0.5, size=3, legend=None, ax=ax, zorder=2)
    sns.boxplot(data=group_df, x='region', y='beta', hue='hemisphere', order=region_order,
               hue_order=HEMISPHERE_ORDER,
               dodge=True, linewidth=1, fliersize=0, ax=ax, zorder=1)
    _outline_only(ax)

    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    sns.move_legend(ax, 'upper left', bbox_to_anchor=(1, 1), title='Hemisphere')
    ax.axhline(y=0, color='0.5', linestyle='--', linewidth=0.5)
    ax.set_ylabel('mean beta')
    ax.set_title(f'contrast-{contrast_label}, group-{group_name}')

    y_top = group_df['beta'].max()
    asterisk_y = _add_asterisk_headroom(ax, y_top)
    sig_rois_set = set(sig_df[
        (sig_df.SNR == contrast_label) & (sig_df.group == group_name) & (sig_df.p_fdr < 0.05)
    ]['region_hemi'])
    for hemi, offset in HEMISPHERE_OFFSET.items():
        for region_idx, region in enumerate(region_order):
            if f'{hemi}-{region}' in sig_rois_set:
                ax.text(region_idx + offset, asterisk_y, '*', ha='center', va='bottom',
                        fontsize=14, fontweight='bold')

    fig.tight_layout()
    sns.despine(ax=ax)
    return fig


# Compare SNRs

## `contrast-snr` for CWNS and CWS

In [ ]:
# get the beta map for the contrast of interest for each subject, both groups combined
all_subs = sub_list_cwns + sub_list_cws
statmap_dict_by_contrast = build_statmap_dict(all_subs, contrast_list, l1_dir)


#### Test plots for a single subject, single ROI

In [ ]:
sub_id = 'sub-SSP008'
mask_ex = os.path.join(masks_dir, sub_id, 'space-MNI152NLin2009cAsym', 'masks-dseg',
                       f'{sub_id}_space-MNI152NLin2009cAsym_mask-R-PT.nii.gz')
mask_ex_img = nib.load(mask_ex)
print(np.unique(mask_ex_img.get_fdata()))


In [ ]:
display = plotting.plot_stat_map(statmap_dict_by_contrast['q'][sub_id],
                                  threshold=1, vmax=10,
                                  cut_coords=[-54, -16, 14])
display.add_contours(mask_ex,
                     filled=False,
                     colors=['k', 'k'])


In [ ]:
region = 'R-PT'
stat_fpath = statmap_dict_by_contrast['q'][sub_id]
mask_fpath = glob(os.path.join(masks_dir, sub_id, f'space-{space_label}', 'masks-dseg', f'*{region}*.nii.gz'))[0]
print(mask_fpath)

mask_img = resample_to_img(mask_fpath, stat_fpath, interpolation='nearest',
                           force_resample=False, copy_header=False)
mask_img = binarize_img(mask_img, two_sided=False)
masked_data = apply_mask(stat_fpath, mask_img)
print(masked_data.mean())


#### Mask stat maps for all ROIs, all contrasts, both groups

In [ ]:
roi_dict = {}
for contrast_label in contrast_list:
    print(f'masking contrast-{contrast_label} stat maps for {len(statmap_dict_by_contrast[contrast_label])} subjects')
    roi_mean_dict = mask_stat_maps(CORTICAL_ROI_LIST, statmap_dict_by_contrast[contrast_label], masks_dir, space_label)
    contrast_df = make_stats_df(roi_mean_dict, group_lookup)
    contrast_df['SNR'] = contrast_label
    roi_dict[contrast_label] = contrast_df

roi_df_long = pd.concat(roi_dict, ignore_index=True)

roi_df_long.SNR = pd.Categorical(roi_df_long.SNR, categories=contrast_list, ordered=True)

# Save the dataframe to a file
with open(os.path.join(group_out_dir, 'roi_df_long.pkl'), 'wb') as f:
    pickle.dump(roi_df_long, f)

roi_df_long.head()


### Statistics

**New**: independent-samples t-test (CWS vs. CWNS) per ROI x contrast, FDR-corrected across ROIs
via `stats_fmt.export_ttests`. **Kept**: within-group one-sample tests (beta != 0), now run for
both groups, also FDR-corrected via `export_ttests`.

In [ ]:
# within-group one-sample tests: is mean beta != 0, per ROI x contrast x group?
records = []
for (region_hemi, snr, group), data in roi_df_long.groupby(['region_hemi', 'SNR', 'group']):
    t_stat, p_val = ttest_1samp(data['beta'], 0)
    records.append({
        'region_hemi': region_hemi, 'SNR': snr, 'group': group,
        't': t_stat, 'df': len(data) - 1, 'p': p_val, 'mean_beta': data['beta'].mean(),
    })

within_group_stats_df = pd.DataFrame(records)
within_group_stats_df['p_fdr'] = multipletests(within_group_stats_df['p'], method='fdr_bh')[1]

export_ttests(within_group_stats_df.to_dict('records'), 'within_group_beta', group_out_dir)
within_group_stats_df.sort_values('p_fdr').head(20)


In [ ]:
# between-group tests: CWS vs. CWNS, per ROI x contrast
records = []
for (region_hemi, snr), data in roi_df_long.groupby(['region_hemi', 'SNR']):
    cws_vals = data.loc[data.group == 'CWS', 'beta']
    cwns_vals = data.loc[data.group == 'CWNS', 'beta']
    if len(cws_vals) < 2 or len(cwns_vals) < 2:
        print(f'Skipping {region_hemi}/{snr}: not enough subjects per group')
        continue

    t_stat, p_val = ttest_ind(cws_vals, cwns_vals)
    records.append({
        'region_hemi': region_hemi, 'SNR': snr,
        't': t_stat, 'df': len(cws_vals) + len(cwns_vals) - 2, 'p': p_val,
        'mean_diff_cws_minus_cwns': cws_vals.mean() - cwns_vals.mean(),
    })

between_group_stats_df = pd.DataFrame(records)
between_group_stats_df['p_fdr'] = multipletests(between_group_stats_df['p'], method='fdr_bh')[1]

export_ttests(between_group_stats_df.to_dict('records'), 'cws_vs_cwns_beta', group_out_dir)
between_group_stats_df.sort_values('p_fdr').head(20)


### Omnibus ANOVA + posthoc (within each group)

One canonical ANOVA + posthoc pass per group -- previously this notebook ran several redundant
`pg.pairwise_tests` blocks that only differed in the order of the `within=[...]` factors, and only
ever on the CWNS group. Left as a within-group structural analysis (hemisphere x region x SNR),
not restructured into a mixed within/between design.

In [ ]:
pd.set_option('display.max_rows', None)

for group_name in ['CWNS', 'CWS']:
    group_long_df = roi_df_long[roi_df_long.group == group_name]

    aov = AnovaRM(group_long_df,
                 aggregate_func='mean',
                 depvar='beta',
                 subject='participant_id',
                 within=['hemisphere', 'SNR', 'region']).fit()
    print(f'--- group: {group_name} ---')
    print(aov)

    pairwise = pg.pairwise_tests(data=group_long_df,
                                 dv='beta',
                                 within=['hemisphere', 'region'],
                                 subject='participant_id',
                                 padjust='fdr')
    pairwise_interaction = pairwise[pairwise.Contrast.str.contains(' * ')].reset_index()
    sig_pairwise = pairwise_interaction[pairwise_interaction['p-corr'] < 0.05].reset_index()
    print(f'Significant {group_name} hemisphere x region interactions (p_FDR < 0.05):')
    print(sig_pairwise[['Contrast', 'hemisphere', 'A', 'B', 'T', 'dof', 'p-unc', 'p-corr', 'BF10']])
    print()


### Box + strip plots

In [ ]:
for contrast_label in contrast_list:
    fig = plot_roi_box_strip(roi_df_long, within_group_stats_df, contrast_label, CORTICAL_ROI_LIST)
    fig.savefig(os.path.join(group_out_dir, f'boxplot_contrast-{contrast_label}.png'))
    fig.savefig(os.path.join(group_out_dir, f'boxplot_contrast-{contrast_label}.svg'))

    for group_name in ['CWS', 'CWNS']:
        fig = plot_roi_box_strip_by_hemisphere(roi_df_long, within_group_stats_df, contrast_label,
                                               group_name, BASE_REGION_ORDER)
        fig.savefig(os.path.join(
            group_out_dir, f'boxplot_contrast-{contrast_label}_group-{group_name}_by-hemisphere.png'))
        fig.savefig(os.path.join(
            group_out_dir, f'boxplot_contrast-{contrast_label}_group-{group_name}_by-hemisphere.svg'))


### Brain plots (surface-based)

Reuses `roi_surface_plotting.py` (copied from `fMRI_auditory-category-learning/`, unmodified) to
project per-ROI scalars onto the fsaverage surface. For each contrast, three panels matching the
CWNS/CWS/diff convention used throughout this pipeline: CWNS-mean beta, CWS-mean beta (both from
the within-group statistics above), and the CWS-CWNS mean difference (from the between-group
statistics above).

In [ ]:
from roi_surface_plotting import plot_roi_surface_stat, build_mask_path_dict

# network_name='dseg' passed directly (the actual mask-directory atlas name), rather than adding
# a new 'cortical' branch to build_mask_path_dict's network_name->mask_network_name mapping --
# avoids touching the copied reference module at all.
mask_path_dict = build_mask_path_dict(CORTICAL_ROI_LIST, masks_dir, network_name='dseg', space_label=space_label)


In [ ]:
# Precompute the per-contrast stat dicts once (fast -- no plotting yet). Splitting the actual
# plotting (slow: surface.vol_to_surf projects every vertex on the whole hemisphere surface, per
# ROI, per call) into separate cells below by comparison type (cws/cwns/diff) gives natural
# checkpoints -- if interrupted partway through, whichever cells already ran have their plots
# saved, and it's clear which comparison type is left to (re-)run.
stat_dicts_by_contrast = {}
for contrast_label in contrast_list:
    cwns_stat_dict = within_group_stats_df[
        (within_group_stats_df.SNR == contrast_label) & (within_group_stats_df.group == 'CWNS')
    ].set_index('region_hemi')['mean_beta'].to_dict()
    cws_stat_dict = within_group_stats_df[
        (within_group_stats_df.SNR == contrast_label) & (within_group_stats_df.group == 'CWS')
    ].set_index('region_hemi')['mean_beta'].to_dict()
    diff_stat_dict = between_group_stats_df[
        between_group_stats_df.SNR == contrast_label
    ].set_index('region_hemi')['mean_diff_cws_minus_cwns'].to_dict()

    stat_dicts_by_contrast[contrast_label] = {'cws': cws_stat_dict, 'cwns': cwns_stat_dict, 'diff': diff_stat_dict}

print('contrasts ready to plot:', list(stat_dicts_by_contrast.keys()))


In [ ]:
def plot_and_save_surface(contrast_label, label, stat_dict):
    if len(stat_dict) == 0:
        print(f'No data for contrast-{contrast_label}, group-{label} -- skipping brain plot.')
        return
    fig = plot_roi_surface_stat(
        stat_dict, mask_path_dict,
        title=f'contrast-{contrast_label}, group-{label}',
    )
    fig.savefig(os.path.join(group_out_dir, f'surface_contrast-{contrast_label}_group-{label}.png'))
    fig.savefig(os.path.join(group_out_dir, f'surface_contrast-{contrast_label}_group-{label}.svg'))


#### CWS surface plots

In [ ]:
for contrast_label in contrast_list:
    plot_and_save_surface(contrast_label, 'cws', stat_dicts_by_contrast[contrast_label]['cws'])


#### CWNS surface plots

In [ ]:
for contrast_label in contrast_list:
    plot_and_save_surface(contrast_label, 'cwns', stat_dicts_by_contrast[contrast_label]['cwns'])


#### CWS − CWNS difference surface plots

In [ ]:
for contrast_label in contrast_list:
    plot_and_save_surface(contrast_label, 'diff', stat_dicts_by_contrast[contrast_label]['diff'])


### Summary

In [ ]:
print(f'ROI beta values: {len(roi_df_long)} rows -> {os.path.join(group_out_dir, "roi_df_long.pkl")}')
print(f'Within-group tests: {len(within_group_stats_df)} ROI x SNR x group tests')
print(f'Between-group (CWS vs. CWNS) tests: {len(between_group_stats_df)} ROI x SNR tests')
print(f'Boxplots and surface brain plots saved to {group_out_dir}')
print()
print('FDR-significant within-group tests, p_FDR < 0.05:')
print(within_group_stats_df[within_group_stats_df.p_fdr < 0.05].sort_values('p_fdr'))
print()
print('FDR-significant CWS vs. CWNS tests, p_FDR < 0.05:')
print(between_group_stats_df[between_group_stats_df.p_fdr < 0.05].sort_values('p_fdr'))
